# Lesson 19: Breakpoints & Time Travel — Replay from Any Checkpoint

## Objective
Use **time travel** to fork graph execution from any past checkpoint, replaying with different inputs to explore "what if" scenarios.

## Limitation of Lesson 18
- ❌ Could only pause/resume forward
- ❌ No way to go back to an earlier step and replay
- ❌ No way to test: "what if I had used different numbers?"

## What is NEW in Lesson 19?
- ✅ **Time travel**: fork from any past checkpoint
- ✅ `get_state_history()` — list all checkpoints for a thread
- ✅ `update_state(config, values, as_node=...)` — modify state at any checkpoint
- ✅ `invoke()` from a specific checkpoint ID
- ✅ Fork pattern: replay from history with modified values

## Theory: Time Travel in LangGraph

```
Thread timeline:
  step 0: {a=5, b=3, op="add"}
  step 1: {result=8}              ← checkpoint saved here
  step 2: {explanation="..."}    ← checkpoint saved here

Time travel: "What if a=10 instead of 5?"
  → Fork from step 0 checkpoint
  → Update state: {a=10, b=3}
  → Re-run from step 0 → step 1 → step 2
  → New thread: {a=10, result=13, explanation="..."}
```

Every `invoke()` call with a checkpointer saves a checkpoint at each step.  
You can branch from ANY of them.


In [ ]:
# ── Cell 1: Imports + LLM Setup ──────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Optional, Literal, Annotated
import operator
from IPython.display import Image, display

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

print("✓ Setup complete")
print("  Time travel requires: MemorySaver (or any checkpointer)")


In [ ]:
# ── Cell 2: State + Nodes ────────────────────────────────────────────────────
class State(TypedDict):
    a: float
    b: float
    operation: str
    result: Optional[float]
    explanation: str
    history: Annotated[list, operator.add]

def parse_node(state: State) -> dict:
    print(f"  [parse]   a={state['a']}, b={state['b']}, op={state['operation']}")
    return {}  # Already parsed in this demo

def compute_node(state: State) -> dict:
    a, b, op = state["a"], state["b"], state["operation"]
    if op == "add":        res, sym = a+b, "+"
    elif op == "multiply": res, sym = a*b, "×"
    else:                  res, sym = (a/b if b else None), "÷"
    entry = f"{a} {sym} {b} = {res}"
    print(f"  [compute] {entry}")
    return {"result": res, "history": [entry]}

def explain_node(state: State) -> dict:
    a, b, op, res = state["a"], state["b"], state["operation"], state["result"]
    prompt = f"In one sentence, explain: {a} {op} {b} = {res}"
    resp = llm.invoke([HumanMessage(content=prompt)])
    expl = resp.content.strip()
    print(f"  [explain] {expl[:80]}")
    return {"explanation": expl, "history": [f"explained: {expl[:40]}..."]}

print("✓ Three-node graph: parse → compute → explain")


In [ ]:
# ── Cell 3: Build Graph ──────────────────────────────────────────────────────
memory = MemorySaver()
builder = StateGraph(State)
builder.add_node("parse",   parse_node)
builder.add_node("compute", compute_node)
builder.add_node("explain", explain_node)
builder.add_edge(START,    "parse")
builder.add_edge("parse",  "compute")
builder.add_edge("compute","explain")
builder.add_edge("explain", END)

graph = builder.compile(checkpointer=memory)
print("✓ Graph with checkpointer compiled")


In [ ]:
# ── Cell 4: Run Original Execution ───────────────────────────────────────────
original_config = {"configurable": {"thread_id": "original-run-1"}}

initial_state = {
    "a": 5.0, "b": 3.0, "operation": "add",
    "result": None, "explanation": "", "history": []
}

print("=== Original Run: 5 + 3 ===")
out = graph.invoke(initial_state, config=original_config)
print(f"\nResult: {out['result']}")
print(f"History: {out['history']}")


In [ ]:
# ── Cell 5: Inspect All Checkpoints ──────────────────────────────────────────
print("All checkpoints for 'original-run-1':")
print()
checkpoints = list(graph.get_state_history(original_config))
for i, ckpt in enumerate(checkpoints):
    print(f"  [{i}] Step {ckpt.metadata.get('step','?'):>2} | "
          f"next={str(ckpt.next):<20} | "
          f"id={ckpt.config['configurable']['checkpoint_id'][:20]}...")

print(f"\nTotal checkpoints: {len(checkpoints)}")


In [ ]:
# ── Cell 6: Time Travel — Fork from compute_node Checkpoint ──────────────────
# Find the checkpoint BEFORE compute_node ran (after parse, before compute)

checkpoints = list(graph.get_state_history(original_config))

# The checkpoint where next=("compute",) is right before compute runs
before_compute = None
for ckpt in checkpoints:
    if "compute" in (ckpt.next or []):
        before_compute = ckpt
        break

if before_compute:
    print(f"Found pre-compute checkpoint:")
    print(f"  ID:     {before_compute.config['configurable']['checkpoint_id'][:30]}...")
    print(f"  State:  a={before_compute.values['a']}, b={before_compute.values['b']}")
    print(f"  Next:   {before_compute.next}")
else:
    print("(checkpoint not found - using most recent)")
    before_compute = checkpoints[-1]


In [ ]:
# ── Cell 7: Fork with Modified State ─────────────────────────────────────────
# "What if a=20 instead of 5?"

# Create a fork config — SAME thread, different checkpoint
fork_config = before_compute.config
fork_config["configurable"]["thread_id"] = "forked-run-1"  # New thread for the fork

# Update the forked state with new values
graph.update_state(
    fork_config,
    {"a": 20.0, "b": 3.0}  # Changed a from 5 to 20
)

print("=== Time Travel Fork: What if a=20 instead of 5? ===")
print("Replaying from compute_node with modified state...")
print()

# Resume from the fork point — will re-run compute and explain with new values
out_fork = graph.invoke(None, config=fork_config)

print(f"\nFork result: {out_fork['result']}")
print(f"Explanation: {out_fork['explanation'][:80]}")
print(f"History:     {out_fork['history']}")
print()
print(f"Original: a=5, b=3 → result={out['result']}")
print(f"Fork:     a=20, b=3 → result={out_fork['result']}")


In [ ]:
# ── Cell 8: Multiple Forks — Explore Different Operations ────────────────────
print("=== Exploring What-If Scenarios ===")
print()

# Fork for each operation
for i, (new_a, new_b, new_op) in enumerate([
    (10.0, 5.0, "add"),
    (10.0, 5.0, "multiply"),
    (10.0, 5.0, "divide"),
]):
    fork_id = f"scenario-{i+1}"
    
    # Get starting checkpoint (before compute)
    cps = list(graph.get_state_history(original_config))
    pre_compute = next((c for c in cps if "compute" in (c.next or [])), cps[-1])
    
    f_config = pre_compute.config.copy()
    f_config["configurable"] = dict(pre_compute.config["configurable"])
    f_config["configurable"]["thread_id"] = fork_id
    
    graph.update_state(f_config, {"a": new_a, "b": new_b, "operation": new_op})
    result = graph.invoke(None, config=f_config)
    
    sym = {"add":"+","multiply":"×","divide":"÷"}.get(new_op,"?")
    print(f"  Scenario {i+1}: {new_a} {sym} {new_b} = {result['result']}")


## Time Travel Concepts

### Checkpoint History
```python
for ckpt in graph.get_state_history(config):
    print(ckpt.config["configurable"]["checkpoint_id"])
    print(ckpt.values)    # State values at this checkpoint
    print(ckpt.next)      # Which node runs next from here
```

### Fork Pattern
```python
# 1. Get the checkpoint you want to fork from
target_ckpt = checkpoints[desired_index]

# 2. Create a config with a NEW thread_id (so you don't overwrite original)
fork_cfg = {**target_ckpt.config, "configurable": {
    **target_ckpt.config["configurable"],
    "thread_id": "my-fork"
}}

# 3. Optionally modify state
graph.update_state(fork_cfg, {"a": new_value})

# 4. Resume from that checkpoint
graph.invoke(None, config=fork_cfg)
```

## Production Use Cases
- ✅ **A/B testing**: try different prompts from the same starting point
- ✅ **Debugging**: replay a crash from the exact failing checkpoint
- ✅ **Experimentation**: test multiple strategies without re-running expensive steps
- ✅ **Auditing**: prove exactly what state the system was in at each step

## Limitations → What Lesson 20 Solves
- ❌ What if the LLM's computed answer is WRONG?
- ❌ How do we make the agent self-correct without human intervention?
- ❌ **Lesson 20**: Self-critique loop — LLM checks its own arithmetic and corrects mistakes
